In [1]:
import sys
sys.path.insert(0, "..")  

from tcr_pipeline.data_io import load_rdata, nested_dict_to_long, long_df_to_wide, save_noise_model, load_noise_model, parse_day
from tcr_pipeline.normalization import library_size_normalize, clr_normalize, tmm_normalize
from tcr_pipeline.visualization import show_traj, plot_noise_model, plot_trajectories
from tcr_pipeline.noise_model import compute_slope_noise_model, compute_stratified_slope_model
from tcr_pipeline.statistics import find_non_neutral
from tcr_pipeline.utils import count_nan_trajectories, impute_trajectories
from tcr_pipeline.nb_glm import fit_nb_dispersion_mle, fit_nb_dispersion_trended, nb_glm_trajectory_test, add_library_sizes, fit_nb_dispersion_trended_smooth
from tcr_pipeline.nb_glm import pool_dispersion_trends_union
import rpy2.robjects as ro
from rpy2.robjects.packages import importr

from tcr_pipeline.edgeR_wrapper import edger_dispersion_trend_pooled, edger_exact_test, edger_trended_dispersion_single_patient

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
healthy_df = pd.read_csv("../data/Healhy-Data-long.csv")

In [3]:
wide = long_df_to_wide(healthy_df)

In [4]:
wide

{'HD106_SEQTR':                   0         46        82        131
 clono                                              
 clono_1      226089.0  238914.0  128798.0  150391.0
 clono_10      37237.0   33193.0   17378.0   19274.0
 clono_100      5035.0    5151.0    6618.0    5272.0
 clono_1000      830.0     826.0     418.0     822.0
 clono_10000     247.0       NaN       NaN       NaN
 ...               ...       ...       ...       ...
 clono_99995      52.0       NaN       NaN       NaN
 clono_99996      52.0       NaN       NaN       NaN
 clono_99997      52.0       NaN       NaN       NaN
 clono_99998      52.0       NaN       NaN       NaN
 clono_99999      52.0       NaN       NaN       NaN
 
 [678329 rows x 4 columns],
 'HD197_SEQTR':                   0         46        82        131
 clono                                              
 clono_1      102708.0  167519.0  164938.0  135860.0
 clono_10      32530.0   41806.0   38454.0   37936.0
 clono_100      4821.0    7905.0    805

In [5]:
healthy_ids = healthy_df['patient'].unique()
nb_models = {}
failed = []

for pid in healthy_ids:
    try:
        nb_models[pid] = fit_nb_dispersion_trended_smooth(healthy_df[healthy_df['patient'] == pid])
    except ValueError as e:
        failed.append((pid, str(e)))

print(f"{len(nb_models)}/{len(healthy_ids)} donors fit successfully")
if failed:
    print("Failed:", failed)

6/6 donors fit successfully


In [6]:
for pid, m in nb_models.items():
    print(pid, m['smooth_x'].min(), m['smooth_x'].max(), m['n_clono_used'])

HD106_SEQTR -5.784817219610194 -4.853760707154171 56441
HD197_SEQTR -5.7431526689359185 -4.881850843855162 65848
HDBR01_SEQTR_ReSeq -5.710071030950582 -4.48622165360617 34567
HDBR02_SEQTR_ReSeq -5.719812535157914 -4.597464957665021 27761
HDBR03_SEQTR_ReSeq -5.69608283134498 -4.535470198049863 67429
HDBR06_SEQTR_ReSeq -5.7199304219067635 -4.228783461450116 19674


In [7]:
nb_pooled = pool_dispersion_trends_union(nb_models)

In [9]:
edger_models = {}
for pid in healthy_ids:
    sub = healthy_df[healthy_df['patient'] == pid]
    if sub['day'].nunique() < 2:
        continue
    edger_df = edger_trended_dispersion_single_patient(sub)
    edger_df['log10_freq'] = edger_df['avg_log_cpm'] * np.log10(2) - 6
    edger_df = edger_df.sort_values('log10_freq')
    edger_models[pid] = {
        'smooth_x': edger_df['log10_freq'].values,
        'smooth_log_phi': -np.log(edger_df['trended_dispersion'].values),
        'global_phi': float(1.0 / edger_df['common_dispersion'].iloc[0]),  # added
    }

edger_pooled = pool_dispersion_trends_union(edger_models)

In [11]:
nb_pooled

{'smooth_x': array([-5.78481722, -5.76909971, -5.75338219, -5.73766468, -5.72194717,
        -5.70622966, -5.69051214, -5.67479463, -5.65907712, -5.64335961,
        -5.62764209, -5.61192458, -5.59620707, -5.58048955, -5.56477204,
        -5.54905453, -5.53333702, -5.5176195 , -5.50190199, -5.48618448,
        -5.47046697, -5.45474945, -5.43903194, -5.42331443, -5.40759691,
        -5.3918794 , -5.37616189, -5.36044438, -5.34472686, -5.32900935,
        -5.31329184, -5.29757433, -5.28185681, -5.2661393 , -5.25042179,
        -5.23470427, -5.21898676, -5.20326925, -5.18755174, -5.17183422,
        -5.15611671, -5.1403992 , -5.12468169, -5.10896417, -5.09324666,
        -5.07752915, -5.06181164, -5.04609412, -5.03037661, -5.0146591 ,
        -4.99894158, -4.98322407, -4.96750656, -4.95178905, -4.93607153,
        -4.92035402, -4.90463651, -4.888919  , -4.87320148, -4.85748397,
        -4.84176646, -4.82604894, -4.81033143, -4.79461392, -4.77889641,
        -4.76317889, -4.74746138, -4.73